# 00 — Kiểm tra môi trường & rủi ro kỹ thuật cao nhất: cài `mamba-ssm` trên Colab T4

**Chạy notebook này đầu tiên, trên Google Colab với Runtime = GPU (T4).**

Mục tiêu (Tuần 1–2 theo đề cương): xác nhận sớm nhất có thể liệu `mamba-ssm`
có build/chạy được trên T4 hay không, và nếu không, xác nhận phương án dự
phòng (Mamba thuần PyTorch, không cần CUDA kernel tùy biến) hoạt động đúng.

Đây là rủi ro cao nhất của toàn bộ đề tài — nếu mục 3 thất bại hoàn toàn,
cần báo GVHD sớm để điều chỉnh phạm vi.

## 1. Kiểm tra GPU & phiên bản torch/CUDA

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version (torch build):", torch.version.cuda)
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))

## 2. Thử cài `mamba-ssm` (thứ tự ưu tiên)

Lỗi build phổ biến trên Colab: version PyTorch/CUDA trên máy ảo Colab thay
đổi thường xuyên và không khớp với wheel prebuilt của `causal-conv1d` /
`mamba-ssm`. Thử lần lượt 3 cách; **ghi lại cách nào thành công** vào
`docs/notes/` để không phải dò lại mỗi lần mở máy Colab mới.

In [ ]:
# Cách 1: pip install thẳng (nhanh nhất nếu có wheel prebuilt khớp version)
!pip install -q mamba-ssm causal-conv1d>=1.4.0

In [ ]:
# Cách 2 (nếu cách 1 lỗi build wheel): cài từ source, tắt build isolation
# để dùng đúng torch đã cài sẵn trên Colab thay vì torch mới nhất từ PyPI.
# Bỏ comment để chạy nếu cần.

# !pip install -q packaging ninja
# !pip install -q --no-build-isolation git+https://github.com/Dao-AILab/causal-conv1d
# !pip install -q --no-build-isolation git+https://github.com/state-spaces/mamba

In [ ]:
# Kiểm tra import + forward pass nhỏ với CUDA kernel thật
try:
    from mamba_ssm import Mamba
    m = Mamba(d_model=64, d_state=16, d_conv=4, expand=2).to("cuda")
    x = torch.randn(2, 32, 64, device="cuda")
    y = m(x)
    print("OK — mamba-ssm (CUDA kernel) hoạt động. Output shape:", y.shape)
    MAMBA_CUDA_OK = True
except Exception as e:
    print("THẤT BẠI — mamba-ssm CUDA kernel không chạy được:")
    print(repr(e))
    MAMBA_CUDA_OK = False

## 3. Phương án dự phòng: Mamba thuần PyTorch (không cần CUDA kernel)

`mamba_ssm` cung cấp `selective_scan_ref` — cài đặt tham chiếu thuần PyTorch,
**chậm hơn** nhưng đúng về mặt số học và không cần biên dịch bất kỳ CUDA
kernel nào. Đây là phương án dự phòng nêu trong đề cương nếu mục 2 thất bại
hoàn toàn trên T4.

In [ ]:
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_ref
    print("OK — selective_scan_ref (thuần PyTorch) import được — có thể dùng làm fallback.")
    FALLBACK_OK = True
except Exception as e:
    print("selective_scan_ref cũng không import được:")
    print(repr(e))
    print("=> Cần cân nhắc phương án khác (vd. cài đặt S4/S6 tối giản từ đầu).")
    FALLBACK_OK = False

## 4. Kết luận rủi ro (ghi lại kết quả để báo cáo GVHD)

In [ ]:
print(f"mamba-ssm CUDA kernel hoạt động trên T4: {MAMBA_CUDA_OK}")
print(f"Fallback thuần PyTorch khả dụng:        {FALLBACK_OK}")
if MAMBA_CUDA_OK:
    print("=> Dùng CUDA kernel thật cho tốc độ train/inference tốt nhất.")
elif FALLBACK_OK:
    print("=> Dùng selective_scan_ref (chậm hơn) — vẫn tiếp tục đề tài đúng kế hoạch,")
    print("   nhưng cần ghi rõ trong báo cáo (Chương 2) và ước lượng lại thời gian train.")
else:
    print("=> RỦI RO NGHIÊM TRỌNG — báo GVHD ngay để điều chỉnh phạm vi đề tài.")

## 5. Cài các thư viện còn lại của pipeline

In [ ]:
!pip install -q librosa soundfile sentencepiece tokenizers datasets jiwer gradio

## 6. (Tuần 3) Thử tải thử VietSuperSpeech — chỉ kiểm tra kết nối, chưa tải toàn bộ

In [ ]:
from datasets import load_dataset

# streaming=True để chỉ xem thử vài mẫu, không tải 267 giờ audio ngay bây giờ
ds = load_dataset("thanhnew2001/VietSuperSpeech", split="train", streaming=True)
sample = next(iter(ds))
print(sample.keys())
print({k: v for k, v in sample.items() if k != "audio"})